In [1]:
from pydantic import BaseModel
from typing import List

class Task(BaseModel):
    id: int
    name: str
    description: str
    agent_type: str    
    url: str 

In [2]:
class TaskPlan(BaseModel):
    user_requirement: str
    tasks: List[Task]

In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [4]:
from google import genai

gemini_client = genai.Client(
    api_key=GOOGLE_API_KEY
)

print("Gemini client created")

Gemini client created


In [6]:
user_requirement = """
Find the best laptop under ₹80,000 for programming,
gaming and college use.
Compare products from Amazon and Flipkart.
"""

In [7]:
def create_plan_gemini(user_requirement: str) -> TaskPlan:
    prompt = f"""
You are the Orchestrator of a shopping research system.

Your job is to understand the user's requirement
and break it into research tasks.

You are NOT doing the research.

For each task determine:

- task id
- task name
- what the task should accomplish
- which type of agent should perform it
- URL to visit if it is a browser task

Available agent types:

browser
llm
logic

Use browser when information must be collected from websites.
Use llm for reasoning, summarization or analysis.
Use logic for deterministic calculations.

For browser tasks:
- provide the exact website URL to visit
- create separate tasks when research should be performed on different websites
- make the task description specific about what information needs to be collected

User requirement:

{user_requirement}
"""
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": TaskPlan.model_json_schema(),
        },
    )
    return TaskPlan.model_validate_json(response.text)

In [8]:
plan = create_plan_gemini(
    "Find the best laptop under ₹80,000 for programming and gaming."
)


In [9]:
for task in plan.tasks:
    print("ID:", task.id)
    print("NAME:", task.name)
    print("TYPE:", task.agent_type)
    print("URL:", task.url)
    print("DESCRIPTION:", task.description)
    print("-" * 50)
  

ID: 1
NAME: Search Amazon India for Gaming Laptops
TYPE: browser
URL: https://www.amazon.in/s?k=gaming+laptop+under+80000
DESCRIPTION: Search for laptops under ₹80,000 suitable for programming and gaming, and collect details such as model name, processor, RAM, GPU, and price.
--------------------------------------------------
ID: 2
NAME: Search Flipkart for Gaming Laptops
TYPE: browser
URL: https://www.flipkart.com/search?q=gaming+laptop+under+80000
DESCRIPTION: Search for laptops under ₹80,000 suitable for programming and gaming, and collect details such as model name, processor, RAM, GPU, and price.
--------------------------------------------------
ID: 3
NAME: Analyze and Compare Laptop Specifications
TYPE: llm
URL: 
DESCRIPTION: Analyze the collected laptop options from Amazon and Flipkart, evaluating their processors, RAM, GPUs, and value for money for both programming and gaming use cases.
--------------------------------------------------
ID: 4
NAME: Calculate Final Price Rankin

In [10]:
import subprocess

In [11]:
import json

browser_task=[]

for task in plan.tasks:
    if task.agent_type == "browser":
        browser_task.append(task.model_dump())
        
browser_task

[{'id': 1,
  'name': 'Search Amazon India for Gaming Laptops',
  'description': 'Search for laptops under ₹80,000 suitable for programming and gaming, and collect details such as model name, processor, RAM, GPU, and price.',
  'agent_type': 'browser',
  'url': 'https://www.amazon.in/s?k=gaming+laptop+under+80000'},
 {'id': 2,
  'name': 'Search Flipkart for Gaming Laptops',
  'description': 'Search for laptops under ₹80,000 suitable for programming and gaming, and collect details such as model name, processor, RAM, GPU, and price.',
  'agent_type': 'browser',
  'url': 'https://www.flipkart.com/search?q=gaming+laptop+under+80000'}]

In [12]:
all_products = []

for task in browser_task:

    result = subprocess.run(
        [
            "python",
            "../agents/product_discovery.py",
            task["url"],
            task["description"]
        ],
        capture_output=True,
        text=True,
        encoding="utf-8"
    )

    # parse result
    products=json.loads(result.stdout)["products"]
    # add products
    all_products.extend(products)
print(all_products)    

[{'name': 'ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB SSD, FHD, 15.6", Windows 11 Home, Graphite Black, 2.3 Kg, FA506NCQ-HN006W', 'price': 73490.0, 'rating': 4.3, 'specifications': 'AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB SSD, FHD, 15.6" 144Hz display, Windows 11', 'url': 'https://www.amazon.in/s?k=gaming+laptop+under+80000', 'source': 'Amazon India'}, {'name': 'ASUS TUF A15 (2025), AMD Ryzen 7 7445HS, Gaming Laptop, RTX 3050-4GB, 75W TGP, 16GB RAM, 1TB SSD, FHD, 15.6", 144Hz, Black, 2.3 Kg, FA506NCG-HN251WS', 'price': 79990.0, 'rating': 4.4, 'specifications': 'AMD Ryzen 7 7445HS, RTX 3050-4GB 75W TGP, 16GB RAM (Upgradeable Upto 64GB), 1TB SSD, FHD, 15.6", 144Hz, Windows 11', 'url': 'https://www.amazon.in/s?k=gaming+laptop+under+80000', 'source': 'Amazon India'}, {'name': 'DELL G15 Intel Core i5 13th Gen 13450HX - (16 GB/512 GB SSD/Windows 11 Home/6 GB Graphics/NVIDIA GeForce RTX 3050)', 'price': 79990.0, 'rating': 4.2, 'specifications': 'Intel Core i5 Processo

In [13]:
import pandas as pd

df = pd.DataFrame(all_products)

df

,name,price,rating,specifications,url,source
0,"ASUS TUF A15, AMD Ryzen 7 170, RTX 3050-4GB, 1...",73490.0,4.3,"AMD Ryzen 7 170, RTX 3050-4GB, 16GB RAM, 512GB...",https://www.amazon.in/s?k=gaming+laptop+under+...,Amazon India
1,"ASUS TUF A15 (2025), AMD Ryzen 7 7445HS, Gamin...",79990.0,4.4,"AMD Ryzen 7 7445HS, RTX 3050-4GB 75W TGP, 16GB...",https://www.amazon.in/s?k=gaming+laptop+under+...,Amazon India
2,DELL G15 Intel Core i5 13th Gen 13450HX - (16 ...,79990.0,4.2,"Intel Core i5 Processor (13th Gen), 16 GB DDR5...",https://www.flipkart.com/search?q=gaming+lapto...,Flipkart
3,Acer Aspire 7 (i7 14th Gen) Intel Core 7 240H ...,79990.0,3.9,"Intel Core 7 Processor, 16 GB DDR4 RAM, 64 bit...",https://www.flipkart.com/search?q=gaming+lapto...,Flipkart
4,MSI Thin 15 Intel Core i7 13th Gen 13620H - (1...,79990.0,4.5,"Intel Core i7 Processor (13th Gen), 16 GB DDR4...",https://www.flipkart.com/search?q=gaming+lapto...,Flipkart
5,HP Victus Intel Core i5 13th Gen 13420H - (16 ...,76888.0,4.2,"Intel Core i5 Processor (13th Gen), 16 GB DDR4...",https://www.flipkart.com/search?q=gaming+lapto...,Flipkart
6,ASUS TUF Gaming F16 (2025) with MSO 2024+M365 ...,79990.0,4.6,"Intel Core 5 Processor, 16 GB DDR5 RAM, Window...",https://www.flipkart.com/search?q=gaming+lapto...,Flipkart


## Leaving the normalization part for now

In [14]:
with open(
    "../outputs/all_products.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        products,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved", len(products), "unique products to ../outputs/unique_products.json")

Saved 5 unique products to ../outputs/unique_products.json


# Youtube Research

In [15]:
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os

load_dotenv(override=True)

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

youtube = build(
    "youtube", 
    "v3", 
    developerKey=YOUTUBE_API_KEY
    )



In [16]:
def search_videos(query,max_no_results=20):

    request=youtube.search().list(
        q=query,
        part="snippet",
        type="video",
        maxResults=max_no_results,
        order="relevance"
    )

    response=request.execute()
    
    videos = []
    for item in response["items"]:
        video_meta_data={
        "video_id":item["id"]["videoId"],
        "title":item["snippet"]["title"],
        "thumbnail_url":item["snippet"]["thumbnails"]["default"]["url"],
        "published": item["snippet"]["publishedAt"],
        "description": item["snippet"]["description"],
        "channel":item["snippet"]["channelTitle"]
        }
        videos.append(video_meta_data)


    return videos



In [17]:
search_videos("Sony WH-1000XM5 review")

[{'video_id': 'aWWHU8-5oqU',
  'title': 'Sony WH 1000XM5 Review - Audiophile Perspective | The Best ANC Headphones?',
  'thumbnail_url': 'https://i.ytimg.com/vi/aWWHU8-5oqU/default.jpg',
  'published': '2022-11-06T06:30:00Z',
  'description': "Sony's 2022 iteration of its popular XM series of headphones is the new WH 1000XM5. I've used this for a while now and here's ...",
  'channel': 'Trakin Tech English'},
 {'video_id': '6CsJZxfZsL0',
  'title': 'Sony WH-1000XM5 Review: Two Steps Forward, One Step Back!',
  'thumbnail_url': 'https://i.ytimg.com/vi/6CsJZxfZsL0/default.jpg',
  'published': '2022-05-12T16:01:05Z',
  'description': "Sony's MK5 noise cancelling headphones are still king of the hill, Sony WH1000XM5: https://geni.us/F6x4cC Sony WH1000XM4: ...",
  'channel': 'Marques Brownlee'},
 {'video_id': 'k_fbRCo-yAs',
  'title': 'Spectacular Headphones With Best ANC | Sony WH-1000XM5',
  'thumbnail_url': 'https://i.ytimg.com/vi/k_fbRCo-yAs/default.jpg',
  'published': '2022-11-26T10:4

In [18]:
def get_video_stats(video_id):
    request=youtube.videos().list(
        part="statistics",
        id=video_id
    )
    response=request.execute()
    stats = response["items"][0]["statistics"]
    video_stats={
        "views": int(stats.get("viewCount", 0)),
        "likes": int(stats.get("likeCount", 0)),
        "comment_count": int(stats.get("commentCount", 0))
    }
    return video_stats


In [19]:
get_video_stats("k_fbRCo-yAs")

{'views': 87651, 'likes': 3521, 'comment_count': 300}

In [20]:
def get_top_comments(video_id, max_results=30):
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        order="relevance",   # gets highest-engagement comments first
        maxResults=max_results,
        textFormat="plainText"
    )
    response = request.execute()
    comments = []
    for item in response["items"]:
        top = item["snippet"]["topLevelComment"]["snippet"]
        comments.append({
            "text": top["textDisplay"],
            "likes": top["likeCount"],
            "author": top["authorDisplayName"]
        })
    return comments

In [21]:
get_top_comments("k_fbRCo-yAs")

[{'text': 'We need that kind of mic test in every video going forward',
  'likes': 14,
  'author': '@Ok_Sounds_Good'},
 {'text': '2:35 "jabbaaa tera ghar bar kidar hai" 😂',
  'likes': 3,
  'author': '@devil-d67'},
 {'text': 'Alreay using these from Last 3 months. Best headphone ever used❤. Beast of ANC🔥',
  'likes': 11,
  'author': '@TechEncoders'},
 {'text': '7:37 KR dollar sign. Bhai apna dollar ka fan 🎉',
  'likes': 1,
  'author': '@caabhisheksingh6300'},
 {'text': "I'm watching this while wearing these headphones. The KING of noise cancellation.❤",
  'likes': 12,
  'author': '@benmetalhead'},
 {'text': 'I loving watching tech reviews from Venom\'s Tech. So I thought of watching the review of a product I already own and I am not disappointed. I am a dialysis patient and I spend 4 hours in dialysis thrice a week, there is too much noise in the dialysis center. So whenever there is too much sound, I turn on the noise-cancelling on this headphone and enjoy the peace "me time". Thank yo

In [22]:
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()

def get_transcript(video_id):
    try:
        fetched_transcript = ytt_api.fetch(video_id)
        # fetched_transcript is iterable of snippets with .text, .start, .duration
        return " ".join(snippet.text for snippet in fetched_transcript)
    except Exception as e:
        print(f"Transcript error for {video_id}: {type(e).__name__}: {e}")
        return None

In [23]:
get_transcript("9a-3r2llDjI")

"so it's been one month since sony launched their new flagship headphones to 1000 x mark 5's and when they came out there was a lot of excitement a lot of claims that these are the best of the best the best wireless headphones you can buy but now that it's been a month i want to talk about what it's actually like to use these as i've been using them in many situations on a day-to-day basis from working out to commuting to working in an office to just hanging out and listening to music and all of those i found some things that are not just based on the spec sheet so i want to spend this video talking a little bit more about the nuances of these headphones to help you decide whether or not they're actually the best pair for you to buy so this is sort of like a day in the life i kind of want to break this down into different categories of like when i was using them so what it's like to use them when you're working out what it's like to use them when you're commuting and i want to break it

In [24]:
query = "Sony WH-1000XM5 review"

videos = search_videos(query, max_no_results=5)

for video in videos:

    print("\n" + "=" * 70)

    print("TITLE:", video["title"])
    print("CHANNEL:", video["channel"])
    print("VIDEO ID:", video["video_id"])

    # Get statistics
    stats = get_video_stats(video["video_id"])

    print("\nSTATS:")
    print("Views:", stats["views"])
    print("Likes:", stats["likes"])
    print("Comments:", stats["comment_count"])

    # Get comments
    comments = get_top_comments(
        video["video_id"],
        max_results=10
    )

    print("\nTOP COMMENTS:")

    for comment in comments:
        print(
            f'{comment["likes"]} likes - '
            f'{comment["author"]}: '
            f'{comment["text"]}'
        )

    # Get transcript
    transcript = get_transcript(video["video_id"])

    print("\nTRANSCRIPT:")

    if transcript:
        print(transcript[:2000])
    else:
        print("Transcript unavailable")


TITLE: Sony WH 1000XM5 Review - Audiophile Perspective | The Best ANC Headphones?
CHANNEL: Trakin Tech English
VIDEO ID: aWWHU8-5oqU

STATS:
Views: 50651
Likes: 1355
Comments: 109

TOP COMMENTS:
1 likes - @tgfan972: waiting for video on "best over/on ear Headphones in every budget "
1 likes - @shilpasankpal7350: Yes I recently purchased it 
It’s voice cancellation feature is amazing . Adaptive voice cancellation is very useful . Comfort of headphones is amazing . I got it for 26k from I world shop
0 likes - @ashunegi6216: bro iam convinced that you have great music taste you listening to TOOL's undertow sober is mind blowing
3 likes - @vasqora: Can you do a over the ear headphone list for under 10k for beginner audiophiles. i basically needed something with a mic and above average audio quality. i checked out reviews for some gaming headphones as well but cant bring my self to trust them, wireless would be appreciated.
3 likes - @abhijitbagchi3471: Please make a playlist, your song re